# Project 2: Fertilizer Optimization Intelligence  
## AI-Driven Precision Agriculture and Resource Optimization

**AAI-510 Final Team Project**  
**Module Owner:** Fertilizer Optimization, Soil Health Intelligence, and Resource Readiness  

---

## Project Purpose

This notebook extends the shared precision agriculture project beyond crop recommendation.  
Where Project 1 answers **"What crop should a farmer plant?"**, this notebook answers:

> **"What fertilizer and soil-health actions are needed to support that crop and improve operational readiness?"**

The goal is to build a graduate-level, production-minded ML workflow that uses soil nutrient and environmental data to support:

- NPK nutrient balancing  
- fertilizer action recommendations  
- soil health scoring  
- sustainability and overuse-risk analysis  
- operational farming readiness  
- explainable AI recommendations  

This keeps the project practical, high-impact, and aligned with real agricultural decision-making.

## Business Problem

Farmers often make fertilizer decisions using experience, fixed schedules, or general recommendations.  
That approach can lead to:

- under-fertilization, which limits productivity  
- over-fertilization, which increases cost and damages soil quality  
- poor nutrient balance across Nitrogen, Phosphorus, and Potassium  
- inefficient irrigation and resource planning  
- weaker operational readiness before planting  

This project uses AI/ML to convert soil and environmental data into actionable fertilizer and soil-health intelligence.

### Business Question

**How can a farmer use soil nutrient and environmental data to determine what fertilizer action is needed before planting or during crop planning?**

## Relationship to Project 1

This notebook is designed as **Project 2** in the larger precision agriculture system.

| Project | Main Question | Primary Output |
|---|---|---|
| Project 1: Intelligent Crop Recommendation | What crop should I plant? | Recommended crop |
| Project 2: Fertilizer Optimization Intelligence | How should I prepare and balance the soil? | Fertilizer and soil-health recommendation |

Together, both modules form a stronger decision-support workflow:

```text
Soil + Weather Conditions
        â†“
Project 1: Crop Recommendation
        â†“
Project 2: Fertilizer Optimization
        â†“
Farmer Action Plan
```

## Dataset

The project uses the **Crop Recommendation Dataset** from Kaggle.

Dataset variables:

| Feature | Description |
|---|---|
| N | Nitrogen level |
| P | Phosphorus level |
| K | Potassium level |
| temperature | Temperature |
| humidity | Humidity |
| ph | Soil pH |
| rainfall | Rainfall |
| label | Crop label |

### Important Limitation

The dataset does **not** include:

- actual fertilizer application records  
- crop yield values  
- farm financial data  
- irrigation schedules  
- soil texture or organic matter  
- regional/seasonal crop calendars  

Therefore, this notebook does **not** claim to predict actual yield or exact fertilizer dosage.  
Instead, it creates an **operational fertilizer recommendation framework** using engineered nutrient balance, crop-specific baselines, and ML classification.

## Notebook Workflow

1. Load dataset  
2. Inspect data quality  
3. Perform fertilizer-focused EDA  
4. Engineer nutrient and soil-health features  
5. Create fertilizer recommendation labels  
6. Train ML models to predict fertilizer action category  
7. Evaluate model performance  
8. Explain model behavior using feature importance and optional SHAP  
9. Generate farmer-facing recommendations  
10. Discuss deployment, monitoring, risks, and business value

In [ ]:
# Core libraries
import os
from pathlib import Path
import subprocess
import sys
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

try:
    import kagglehub
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
    import kagglehub

# Visualization
import matplotlib.pyplot as plt

# ML
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Model persistence
import joblib

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load Dataset

This notebook first looks for Andrea's engineered dataset.  
If it is not available, it falls back to the raw Kaggle dataset using the same `kagglehub` source as `feature_engineering.ipynb`.

### Local and Google Colab Compatibility Notes

- In Google Colab, run the import/setup cell first; it installs `kagglehub` if needed.
- Locally, the notebook uses the active Jupyter Python environment and downloads the Kaggle dataset through `kagglehub` if no local CSV exists.
- No Kaggle API token is required for this public dataset through `kagglehub`.
- If a local processed file exists, the notebook uses that first so Project 2 can reuse Project 1 or feature-engineering outputs.

Expected possible local files:

```text
data/processed/engineered_ag_data.csv
data/processed/crop_recommendation_clean.csv
data/raw/Crop_recommendation.csv
Crop_recommendation.csv
```


In [ ]:
# Candidate local dataset paths
candidate_paths = [
    Path("data/processed/engineered_ag_data.csv"),
    Path("data/processed/crop_recommendation_clean.csv"),
    Path("data/raw/Crop_recommendation.csv"),
    Path("Crop_recommendation.csv")
]

data_path = None
for path in candidate_paths:
    if path.exists():
        data_path = path
        break

if data_path is None:
    print("Local dataset not found. Downloading Crop Recommendation Dataset from Kaggle...")
    kaggle_path = Path(kagglehub.dataset_download("atharvaingle/crop-recommendation-dataset"))
    csv_files = sorted(kaggle_path.glob("*.csv"))

    if not csv_files:
        raise FileNotFoundError(
            f"Kaggle dataset downloaded to {kaggle_path}, but no CSV file was found."
        )

    data_path = csv_files[0]

df = pd.read_csv(data_path)
print(f"Loaded dataset from: {data_path}")
print(f"Shape: {df.shape}")
df.head()

## 2. Standardize Column Names and Validate Required Fields

This keeps the notebook resilient if column names differ slightly across the raw and processed versions.

In [ ]:
# Standardize column names
df.columns = [col.strip().lower() for col in df.columns]

required_cols = ["n", "p", "k", "temperature", "humidity", "ph", "rainfall", "label"]
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Remove duplicates if present
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
after = len(df)

print(f"Rows before duplicate removal: {before}")
print(f"Rows after duplicate removal: {after}")
df.info()

## 3. Data Quality Review

Before building recommendations, we confirm whether the dataset has missing values, invalid values, or class imbalance.

In [ ]:
# Missing values
missing_summary = df[required_cols].isna().sum().to_frame("missing_count")
missing_summary["missing_pct"] = (missing_summary["missing_count"] / len(df) * 100).round(2)
missing_summary

In [ ]:
# Basic numeric summary
df[["n", "p", "k", "temperature", "humidity", "ph", "rainfall"]].describe().T

In [ ]:
# Crop label distribution
crop_counts = df["label"].value_counts().sort_values(ascending=False)
crop_counts

### Initial Data Interpretation

The raw features are directly relevant to agricultural operations:

- **N, P, K** support fertilizer and nutrient balance decisions.  
- **pH** affects nutrient availability and soil suitability.  
- **Rainfall, humidity, and temperature** support water readiness and environmental suitability.  
- **Crop labels** allow crop-specific nutrient baselines to be calculated from the dataset.  

The fertilizer optimization layer will use both the raw variables and engineered indicators.

## 4. Fertilizer-Focused EDA

The goal is not just to visualize data.  
The goal is to understand how nutrient levels vary across crops and how those differences can support fertilizer strategy.

In [ ]:
# Average NPK requirement by crop label
crop_npk_profile = (
    df.groupby("label")[["n", "p", "k"]]
    .mean()
    .round(2)
    .sort_index()
)

crop_npk_profile.head()

In [ ]:
# Plot average NPK by crop
ax = crop_npk_profile.plot(kind="bar", figsize=(14, 6))
ax.set_title("Average NPK Levels by Crop")
ax.set_xlabel("Crop")
ax.set_ylabel("Average Nutrient Level")
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()

In [ ]:
# pH distribution by crop
crop_ph_profile = (
    df.groupby("label")["ph"]
    .agg(["mean", "min", "max"])
    .round(2)
    .sort_values("mean")
)
crop_ph_profile.head(10)

In [ ]:
ax = crop_ph_profile["mean"].plot(kind="bar", figsize=(14, 5))
ax.set_title("Average Soil pH by Crop")
ax.set_xlabel("Crop")
ax.set_ylabel("Average pH")
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()

In [ ]:
# Rainfall and humidity by crop
crop_water_profile = (
    df.groupby("label")[["rainfall", "humidity", "temperature"]]
    .mean()
    .round(2)
    .sort_values("rainfall", ascending=False)
)
crop_water_profile.head()

In [ ]:
ax = crop_water_profile[["rainfall", "humidity"]].plot(kind="bar", figsize=(14, 6))
ax.set_title("Average Rainfall and Humidity by Crop")
ax.set_xlabel("Crop")
ax.set_ylabel("Average Value")
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()

### EDA Insight

Different crops show different nutrient and water profiles.  
That supports a crop-aware fertilizer strategy instead of one generic fertilizer recommendation for every farm condition.

This is the first key business insight:

> Fertilizer strategy should depend on both current NPK levels and the crop being targeted.

## 5. Feature Engineering

This section converts raw soil and weather values into decision-ready agricultural indicators.

Feature engineering includes:

- nutrient ratios  
- total nutrient index  
- pH suitability indicator  
- water availability score  
- nutrient gap against crop-specific baselines  
- deficiency/excess flags  
- soil health score  
- fertilizer priority score

In [ ]:
# Avoid division by zero with a small epsilon
EPS = 1e-6

df_fe = df.copy()

# Nutrient ratio features
df_fe["np_ratio"] = df_fe["n"] / (df_fe["p"] + EPS)
df_fe["nk_ratio"] = df_fe["n"] / (df_fe["k"] + EPS)
df_fe["pk_ratio"] = df_fe["p"] / (df_fe["k"] + EPS)

# Total nutrient availability
df_fe["total_npk"] = df_fe["n"] + df_fe["p"] + df_fe["k"]

# Environmental support features
df_fe["water_availability_index"] = (
    0.6 * df_fe["rainfall"].rank(pct=True) +
    0.4 * df_fe["humidity"].rank(pct=True)
)

# Heat stress proxy: high temperature + lower humidity
df_fe["heat_stress_index"] = (
    df_fe["temperature"].rank(pct=True) * (1 - df_fe["humidity"].rank(pct=True))
)

# pH suitability proxy: closer to neutral/slightly acidic range is usually more favorable
# This is a general agronomic proxy, not a crop-specific fertilizer prescription.
df_fe["ph_distance_from_neutral"] = np.abs(df_fe["ph"] - 6.5)
df_fe["ph_suitability_score"] = 1 - (df_fe["ph_distance_from_neutral"] / df_fe["ph_distance_from_neutral"].max())
df_fe["ph_suitability_score"] = df_fe["ph_suitability_score"].clip(0, 1)

df_fe.head()

## 6. Crop-Specific Nutrient Baselines

The dataset does not include official fertilizer prescriptions.  
To stay honest, this notebook derives **crop-specific nutrient baselines** from the observed dataset averages.

These baselines are used to estimate whether a sample is low, balanced, or high relative to the nutrient profile commonly associated with that crop label.

In [ ]:
# Crop-specific nutrient averages
nutrient_baselines = (
    df_fe.groupby("label")[["n", "p", "k"]]
    .mean()
    .rename(columns={
        "n": "ideal_n",
        "p": "ideal_p",
        "k": "ideal_k"
    })
)

df_fe = df_fe.merge(nutrient_baselines, on="label", how="left")

# Nutrient gaps relative to crop-specific baseline
df_fe["n_gap"] = df_fe["ideal_n"] - df_fe["n"]
df_fe["p_gap"] = df_fe["ideal_p"] - df_fe["p"]
df_fe["k_gap"] = df_fe["ideal_k"] - df_fe["k"]

# Absolute imbalance
df_fe["total_abs_nutrient_gap"] = (
    df_fe["n_gap"].abs() + df_fe["p_gap"].abs() + df_fe["k_gap"].abs()
)

df_fe[["label", "n", "p", "k", "ideal_n", "ideal_p", "ideal_k", "n_gap", "p_gap", "k_gap"]].head()

## 7. Nutrient Status Flags

For each nutrient, create operational status categories:

- **low**: nutrient appears below crop-specific baseline  
- **balanced**: nutrient appears close to crop-specific baseline  
- **high**: nutrient appears above crop-specific baseline  

The tolerance controls how strict the system is.

In [ ]:
# Tolerance as percentage of crop-specific ideal nutrient value
TOLERANCE = 0.15

def nutrient_status(actual, ideal, tolerance=TOLERANCE):
    lower = ideal * (1 - tolerance)
    upper = ideal * (1 + tolerance)
    if actual < lower:
        return "low"
    elif actual > upper:
        return "high"
    else:
        return "balanced"

for nutrient in ["n", "p", "k"]:
    df_fe[f"{nutrient}_status"] = df_fe.apply(
        lambda row: nutrient_status(row[nutrient], row[f"ideal_{nutrient}"]),
        axis=1
    )

df_fe[["label", "n_status", "p_status", "k_status"]].head()

In [ ]:
# Status distribution
status_summary = {}
for nutrient in ["n", "p", "k"]:
    status_summary[nutrient.upper()] = df_fe[f"{nutrient}_status"].value_counts()

pd.DataFrame(status_summary).fillna(0).astype(int)

## 8. Fertilizer Recommendation Label

This creates the target label for the fertilizer intelligence model.

The logic is intentionally transparent:

- If Nitrogen is low, recommend increasing Nitrogen.  
- If Phosphorus is low, recommend increasing Phosphorus.  
- If Potassium is low, recommend increasing Potassium.  
- If multiple nutrients are low, recommend NPK rebalance.  
- If no nutrients are low but one is high, recommend reducing over-application risk.  
- If nutrients are balanced, maintain current strategy.  

This label is not an official agronomic prescription.  
It is an operational recommendation class for ML demonstration and decision-support design.

In [ ]:
def fertilizer_recommendation(row):
    low_nutrients = []
    high_nutrients = []

    for nutrient in ["n", "p", "k"]:
        status = row[f"{nutrient}_status"]
        if status == "low":
            low_nutrients.append(nutrient.upper())
        elif status == "high":
            high_nutrients.append(nutrient.upper())

    if len(low_nutrients) >= 2:
        return "rebalance_npk"
    elif len(low_nutrients) == 1:
        return f"increase_{low_nutrients[0]}"
    elif len(high_nutrients) >= 2:
        return "reduce_overapplication"
    elif len(high_nutrients) == 1:
        return f"monitor_high_{high_nutrients[0]}"
    else:
        return "maintain_current_levels"

df_fe["fertilizer_action"] = df_fe.apply(fertilizer_recommendation, axis=1)

df_fe["fertilizer_action"].value_counts()

## 9. Soil Health and Resource Readiness Scores

These are business-friendly indicators for dashboards and executive communication.

They help translate technical feature engineering into farmer-facing decision support.

In [ ]:
# Normalize helper
def minmax(series):
    return (series - series.min()) / (series.max() - series.min() + EPS)

# Nutrient balance score: lower nutrient gap means better balance
df_fe["nutrient_balance_score"] = 1 - minmax(df_fe["total_abs_nutrient_gap"])
df_fe["nutrient_balance_score"] = df_fe["nutrient_balance_score"].clip(0, 1)

# Soil health score combines nutrient balance and pH suitability
df_fe["soil_health_score"] = (
    0.70 * df_fe["nutrient_balance_score"] +
    0.30 * df_fe["ph_suitability_score"]
) * 100

# Resource readiness combines soil health and water availability
df_fe["farm_readiness_score"] = (
    0.60 * (df_fe["soil_health_score"] / 100) +
    0.40 * df_fe["water_availability_index"]
) * 100

# Fertilizer priority: high nutrient gap and weaker soil health means higher priority
df_fe["fertilizer_priority_score"] = (
    0.70 * minmax(df_fe["total_abs_nutrient_gap"]) +
    0.30 * (1 - df_fe["soil_health_score"] / 100)
) * 100

df_fe[[
    "label", "soil_health_score", "farm_readiness_score", 
    "fertilizer_priority_score", "fertilizer_action"
]].head()

In [ ]:
# Categorize soil health
def score_category(score):
    if score >= 75:
        return "healthy"
    elif score >= 50:
        return "moderate"
    else:
        return "needs_attention"

df_fe["soil_health_category"] = df_fe["soil_health_score"].apply(score_category)
df_fe["farm_readiness_category"] = df_fe["farm_readiness_score"].apply(score_category)

df_fe[["soil_health_category", "farm_readiness_category"]].value_counts().reset_index(name="count").head()

## 10. Modeling Objective

We will train machine learning models to predict the **fertilizer action category** from soil, nutrient, and environmental features.

### Target

```text
fertilizer_action
```

### Why this matters

In a production setting, this model could support:

- fertilizer planning  
- agronomist decision support  
- input procurement planning  
- sustainability monitoring  
- farm operational readiness scoring

## Modeling Integrity Note

The `fertilizer_action` target in this notebook is an engineered decision-support class derived from nutrient status rules.  
That means model accuracy measures how well the ML models reproduce the transparent fertilizer-action framework, not whether the recommendations have been validated by field trials.

This keeps the notebook aligned with the project goal: build a practical fertilizer optimization intelligence layer while clearly avoiding unsupported claims about exact fertilizer dosage, yield prediction, or production agronomic readiness.


In [ ]:
# Select model features
feature_cols = [
    "n", "p", "k", "temperature", "humidity", "ph", "rainfall",
    "np_ratio", "nk_ratio", "pk_ratio", "total_npk",
    "water_availability_index", "heat_stress_index", 
    "ph_distance_from_neutral", "ph_suitability_score",
    "n_gap", "p_gap", "k_gap", "total_abs_nutrient_gap",
    "nutrient_balance_score", "soil_health_score", 
    "farm_readiness_score", "fertilizer_priority_score"
]

# Keep only columns that exist, in case Andrea's notebook already included variants
feature_cols = [col for col in feature_cols if col in df_fe.columns]

X = df_fe[feature_cols]
y = df_fe["fertilizer_action"]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Feature count:", len(feature_cols))
print("Target classes:", list(label_encoder.classes_))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_encoded
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

## 11. Train and Compare Models

The project should not use only one model.  
A graduate-level ML workflow should compare a baseline model against stronger nonlinear/ensemble models.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=8,
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=RANDOM_STATE
    )
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, preds, average="weighted", zero_division=0
    )

    results.append({
        "model": name,
        "accuracy": acc,
        "precision_weighted": precision,
        "recall_weighted": recall,
        "f1_weighted": f1
    })

results_df = pd.DataFrame(results).sort_values("f1_weighted", ascending=False)
results_df

In [ ]:
ax = results_df.set_index("model")[["accuracy", "f1_weighted"]].plot(kind="bar", figsize=(10, 5))
ax.set_title("Model Comparison: Fertilizer Action Prediction")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 12. Select Final Model

The final model should balance:

- predictive performance  
- interpretability  
- operational usefulness  
- deployment simplicity  

For this project, an ensemble model such as Random Forest is a strong candidate because it handles nonlinear relationships well and provides useful feature importance.

In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_model = models[best_model_name]

print(f"Best model selected: {best_model_name}")

y_pred = best_model.predict(X_test)

print(classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_,
    zero_division=0
))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(cm)

ax.set_title(f"Confusion Matrix - {best_model_name}")
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")

ax.set_xticks(np.arange(len(label_encoder.classes_)))
ax.set_yticks(np.arange(len(label_encoder.classes_)))
ax.set_xticklabels(label_encoder.classes_, rotation=45, ha="right")
ax.set_yticklabels(label_encoder.classes_)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()

## 13. Feature Importance

Feature importance helps explain which soil and environmental indicators drive fertilizer action predictions.

This supports trust and makes the model easier to explain to technical and nontechnical audiences.

In [ ]:
# Extract feature importance if available
def get_feature_importance(model, feature_names):
    # Pipeline case
    if hasattr(model, "named_steps"):
        estimator = model.named_steps.get("model")
    else:
        estimator = model

    if hasattr(estimator, "feature_importances_"):
        return pd.DataFrame({
            "feature": feature_names,
            "importance": estimator.feature_importances_
        }).sort_values("importance", ascending=False)
    elif hasattr(estimator, "coef_"):
        coef = np.abs(estimator.coef_).mean(axis=0)
        return pd.DataFrame({
            "feature": feature_names,
            "importance": coef
        }).sort_values("importance", ascending=False)
    else:
        return None

importance_df = get_feature_importance(best_model, feature_cols)

if importance_df is not None:
    display(importance_df.head(15))
else:
    print("Selected model does not expose standard feature importance.")

In [ ]:
if importance_df is not None:
    ax = importance_df.head(15).sort_values("importance").plot(
        kind="barh",
        x="feature",
        y="importance",
        legend=False,
        figsize=(10, 6)
    )
    ax.set_title(f"Top Feature Drivers - {best_model_name}")
    ax.set_xlabel("Importance")
    ax.set_ylabel("Feature")
    plt.tight_layout()
    plt.show()

## 14. Optional SHAP Explainability

Run this section if SHAP is installed.

SHAP can help explain individual fertilizer recommendations and support trust in the decision-support system.

In [ ]:
try:
    import shap

    # SHAP works best here with tree-based models.
    if hasattr(best_model, "named_steps"):
        shap_model = best_model.named_steps.get("model")
        X_explain = best_model.named_steps.get("scaler").transform(X_test) if "scaler" in best_model.named_steps else X_test
    else:
        shap_model = best_model
        X_explain = X_test

    if best_model_name in ["Decision Tree", "Random Forest", "Gradient Boosting"]:
        explainer = shap.Explainer(shap_model, X_test)
        shap_values = explainer(X_test.iloc[:100])
        shap.plots.beeswarm(shap_values, max_display=15)
    else:
        print("SHAP section skipped for this model type. Use Random Forest or Gradient Boosting for best results.")

except Exception as e:
    print("SHAP is not installed or could not run in this environment.")
    print("Install with: pip install shap")
    print("Error:", e)

## 15. Farmer-Facing Recommendation Function

This function converts model predictions into plain-language operational guidance.

This is important because the final solution should not only output a class label.  
It should provide actionable recommendations.

In [ ]:
def generate_fertilizer_guidance(row, model=best_model):
    # Prepare one-row DataFrame
    input_df = pd.DataFrame([row])[feature_cols]

    pred_encoded = model.predict(input_df)[0]
    pred_action = label_encoder.inverse_transform([pred_encoded])[0]

    guidance_map = {
        "increase_N": "Increase nitrogen application to improve nutrient balance.",
        "increase_P": "Increase phosphorus application to support root development and crop readiness.",
        "increase_K": "Increase potassium application to support crop resilience and plant health.",
        "rebalance_npk": "Rebalance NPK fertilizer strategy because multiple nutrients appear below crop-specific baseline.",
        "reduce_overapplication": "Review fertilizer application because multiple nutrients appear above crop-specific baseline.",
        "monitor_high_N": "Monitor nitrogen levels to avoid over-application and unnecessary fertilizer cost.",
        "monitor_high_P": "Monitor phosphorus levels to reduce runoff and soil imbalance risk.",
        "monitor_high_K": "Monitor potassium levels to avoid nutrient imbalance.",
        "maintain_current_levels": "Maintain current nutrient strategy. Soil nutrient balance appears acceptable."
    }

    return {
        "crop": row.get("label", "unknown"),
        "fertilizer_action": pred_action,
        "guidance": guidance_map.get(pred_action, "Review soil condition and consult agronomic guidance."),
        "soil_health_score": round(row.get("soil_health_score", np.nan), 2),
        "farm_readiness_score": round(row.get("farm_readiness_score", np.nan), 2),
        "fertilizer_priority_score": round(row.get("fertilizer_priority_score", np.nan), 2)
    }

# Example recommendation
example_row = df_fe.iloc[0].to_dict()
generate_fertilizer_guidance(example_row)

## 16. Sample Operational Output

Create a small farmer-facing table that combines crop, nutrient status, fertilizer action, and readiness scores.

In [ ]:
sample_outputs = []

for _, row in df_fe.sample(10, random_state=RANDOM_STATE).iterrows():
    result = generate_fertilizer_guidance(row.to_dict())
    result.update({
        "N_status": row["n_status"],
        "P_status": row["p_status"],
        "K_status": row["k_status"],
    })
    sample_outputs.append(result)

recommendation_table = pd.DataFrame(sample_outputs)
recommendation_table

## 17. Save Engineered Dataset and Model Artifacts

A production-minded project should preserve outputs for reuse.

This supports:

- reproducibility  
- GitHub quality  
- deployment discussion  
- future inference workflow

In [ ]:
# Create output directories
Path("data/processed").mkdir(parents=True, exist_ok=True)
Path("models").mkdir(parents=True, exist_ok=True)

# Save engineered data
engineered_output_path = Path("data/processed/fertilizer_intelligence_features.csv")
df_fe.to_csv(engineered_output_path, index=False)

# Save model and supporting objects
model_output_path = Path("models/fertilizer_action_model.joblib")
encoder_output_path = Path("models/fertilizer_action_label_encoder.joblib")
features_output_path = Path("models/fertilizer_feature_columns.joblib")

joblib.dump(best_model, model_output_path)
joblib.dump(label_encoder, encoder_output_path)
joblib.dump(feature_cols, features_output_path)

print(f"Saved engineered dataset to: {engineered_output_path}")
print(f"Saved model to: {model_output_path}")
print(f"Saved label encoder to: {encoder_output_path}")
print(f"Saved feature columns to: {features_output_path}")

## 18. Deployment Strategy

A realistic deployment does not require a full application for this course, but the deployment plan should be credible.

### Proposed Deployment Workflow

```text
Farmer / Sensor Input
        â†“
Soil + Weather Data Validation
        â†“
Feature Engineering Pipeline
        â†“
Crop Recommendation Module
        â†“
Fertilizer Optimization Module
        â†“
Soil Health + Readiness Scores
        â†“
Farmer Recommendation Dashboard or Mobile App
```

### Batch vs Real-Time

- **Batch mode:** seasonal planning, farm-level fertilizer procurement, crop planning  
- **Real-time / near-real-time mode:** IoT soil sensor updates, rainfall changes, irrigation readiness alerts  

### Hosting

Potential hosting options:

- cloud API for cooperative/farm-management systems  
- lightweight edge/mobile model for rural use  
- dashboard connected to soil testing data  

### Monitoring

The deployed system should monitor:

- data drift across regions or seasons  
- changes in soil nutrient distributions  
- model confidence by crop type  
- recommendation error feedback from farmers or agronomists  
- fertilizer overuse and sustainability indicators

## 19. Ethical, Business, and Regulatory Risks

### Risk 1: Overconfidence in Recommendations

The model should not replace agronomists or local farming expertise.  
It should support decision-making.

### Risk 2: Dataset Generalization

The dataset may not represent every region, soil type, climate zone, or crop variety.  
Deployment would require local calibration.

### Risk 3: Environmental Impact

Incorrect fertilizer recommendations could increase runoff, soil degradation, or unnecessary cost.  
The system should include sustainability safeguards.

### Risk 4: Farmer Accessibility

A solution that requires constant internet or expensive sensors may not be accessible to smallholder farmers.

### Risk 5: Missing Yield Data

The dataset does not include actual yield outcomes.  
The project should discuss productivity readiness, not claim direct yield prediction.

## 20. Business Recommendations

Based on this project design, a practical implementation should:

1. Start as a decision-support tool for crop planning and soil preparation.  
2. Use crop recommendation and fertilizer optimization together.  
3. Add local agronomic calibration before real deployment.  
4. Include human review for fertilizer recommendations.  
5. Track recommendation outcomes over time to improve the model.  
6. Integrate with soil testing services or IoT sensors when available.  

### Executive Summary

This fertilizer optimization module converts simple soil and weather measurements into actionable nutrient intelligence.  
It helps farmers understand not only what crop may fit the land, but what soil preparation actions may be needed to support that crop.

## 21. Conclusion

This notebook extends the project from basic crop recommendation into a more complete precision agriculture decision-support system.

The main contribution is the **Fertilizer Optimization Intelligence** layer, which provides:

- nutrient balance analysis  
- crop-aware NPK gap estimation  
- fertilizer action classification  
- soil health scoring  
- resource readiness scoring  
- explainable recommendations  
- realistic deployment planning  

This strengthens the final team project by connecting machine learning outputs to operational farming decisions, sustainability, and resource readiness.